# 02 — Per-person effects (and later, the budgeted plan)

For each holdout customer we want two numbers: extra dollars from the men's email vs nothing, and from the women's email vs nothing. That is a **difference**, not “will they buy anyway.”

Models are trained on the **build** half only. We check whether their *average* predicted extra spend recovers the experiment's measured average — we do **not** add up each model's own guesses as a score.

In [ ]:
from causal_uplift.data import load_and_split
from causal_uplift.effects import fit_all_cate_models, measured_ate, recovery_table

build, grade = load_and_split()
print(f"build={len(build):,}  grade={len(grade):,}")
print("measured ATE on the grade set:", measured_ate(grade))

results = fit_all_cate_models(build, grade)
recovery_table(results, grade)

## What to look for

- Average predicted extra spend should be near the measured averages (about $0.40 women's, about $0.60–$0.80 men's on a typical split — noisy because spend is mostly zeros).
- Almost nobody should look like they are *hurt* by an email on this dataset.
- The causal forest also reports, for each person, a range. The share whose range stays entirely above zero is a later check that the pattern is real.

Turning these numbers into a send list under the budget is the next section.

## The budgeted plan

For each person and email: extra spend minus cost. Skip anything that is not worth sending. Rank the rest by **profit per dollar**, one email per person, until the $2,400 is gone.

We grade those send lists with the fair report card (real holdout outcomes), not by adding up the model's own guesses.

In [ ]:
from causal_uplift.allocation import run_allocation

table = run_allocation()
table[["policy", "n_womens", "n_mens", "n_control", "spent", "aipw", "aipw_ci_low", "aipw_ci_high"]]